# Chapter 17 — Control Flow

Part 7: Kernel construction and optimization.

**Ladder:** scalar branch and Python loop (Method 01) → mask-based lane execution and `tl.static_range`/`range` kernels (Method 02) → compile-time specialization plus dynamic-shape/grid-stride kernels (Method 03).

**Thread.**

- **Recurring idea.** Control flow is a type decision: a `constexpr` condition becomes a compiled branch; a uniform runtime scalar becomes a device branch; a per-lane tensor becomes a mask or `tl.where`.
- **Chain.** §17.1 `if` and `constexpr` → §17.2 `tl.static_range` → §17.3 `range` → §17.4 unrolling → §17.5 conditional execution → §17.6 mask-based execution → §17.7 specialization → §17.8 dynamic shapes.
- **Payoff.** The same decision explains loop unrolling, specialization keys, and the static-block/dynamic-`N` pattern used by production kernels.

**Study tip.**

- Fix the spine $x=(1,2,3,4,5,6)$ and threshold $T=3$.
- Write the mask $m=[x\gt T]$, **both** arms, and the selected vector before reading code.
- For a partial tile with `BLOCK=4` and $N=6$, program 1 owns offsets $4,5,6,7$ and must mask lanes $6,7$.

| item | value |
|---|---|
| $m=[x\gt T]$ | $(\mathrm F,\mathrm F,\mathrm F,\mathrm T,\mathrm T,\mathrm T)$ |
| arms $x$ / $0$ | $(1,2,3,4,5,6)$ / $(0,0,0,0,0,0)$ |
| selected | $(0,0,0,4,5,6)$ |

**Learning objectives**

- Classify every condition as **compile-time** (`tl.constexpr`), **uniform runtime** (a scalar program id or kernel argument), or **per-lane** (a tensor).
- Explain why a Python `if` on a `tl.constexpr` is resolved at compile time, while a data-dependent condition must become a mask or `tl.where`.
- Choose `tl.static_range` for a small fixed trip count and `range` for a dynamic one, and state the code-size/register trade-off.
- Read and write mask tables for lane selection and partial tiles, and separate a uniform branch from divergent per-lane execution.
- Predict when a launch creates a new specialization and why passing a dynamic size as `constexpr` causes recompilation.
- Verify every rule with an `assert` or `torch.testing.assert_close`, not by reading printed tables.

[Index](../../00_index.ipynb) · [Chapter README](README.md) · [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)


## Chapter front matter

[Index](../../00_index.ipynb) · [← Previous](../../part6_reductions/ch16_combining_ops/16_complete.ipynb) · [Next →](../ch18_kernel_construction/18_complete.ipynb)

**Estimated time:** 55–70 min · **Difficulty:** Intermediate

**Prerequisites**

- Chapter 1: `@triton.jit`, compile-time versus runtime arguments, and `tl.constexpr`.
- Chapter 5: masked `tl.load` / `tl.store` and `other=`.
- Chapter 6: offsets, tiles, and boundary masks.
- Chapter 8: comparison masks, `&` versus `and`, and `tl.where`.
- No prior GPU control-flow background is assumed.

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) · [Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) · [Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) · [Troubleshooting](../../docs/TROUBLESHOOTING.md) · [Exercise hints](../../docs/SOLUTIONS.md)


## Concept diagrams

### Scalar `if` versus tensor mask

```text
scalar if (uniform)                 tensor mask (per lane)
-------------------                 ----------------------
if pid == 0:                        m = x > T          # one bit per lane
    y = A                           y = tl.where(m, A, B)
else:
    y = B                           both arms are evaluated;
                                    m selects lane by lane
```

- A scalar branch is taken once per program.
- A tensor condition cannot be a Python `if`; it is a predicate tile.
- The mask keeps every lane on the same instruction stream, so there is no divergence.

### Static versus dynamic loop

```text
tl.static_range(0, 4)               range(0, n_steps)
---------------------               -----------------
body emitted 4 times                body emitted once
no loop branch                      one compare/branch per trip
trip count must be constexpr        trip count may be a runtime scalar
code size grows with 4              code size is constant in n_steps
```

- The static loop trades code size for zero loop branches.
- The dynamic loop keeps one body and pays one compare/branch per trip.

### Unrolling

```text
loop:      y0 -> [body] -> y1 -> [body] -> y2 -> [body] -> y3 -> [body] -> y4
unrolled:  y0 -> [body][body][body][body] -> y4
```

- Unrolling removes the back-edge and exposes independent work.
- It replicates instructions and can raise register pressure.

### Specialization

```text
launch(MODE=0, BLOCK=4)          -> binary A
launch(MODE=1, BLOCK=4)          -> binary B
launch(MODE=0, BLOCK=8)          -> binary C
launch(MODE=0, BLOCK=4, n=17)    -> binary A (n is runtime; no new binary)
```

- The `constexpr` tuple is part of the cache key.
- The runtime scalar `n` is not.


In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import torch
import triton
import triton.language as tl

from ttools import DEVICE, HAS_GPU, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU, "| triton:", triton.__version__)


## Part I — Mathematics and concepts

**Thread.**

- **One decision rule.** The compiler knows → branch; the lane knows → mask; the launch knows → runtime scalar plus a tail mask.
- **Order.** The eight subsections move from the condition to the loop bound, then to execution uniformity, specialization, and shape.

### 17.1 `if` and compile-time conditions

**Connection.** Chapter 16 combined reductions and scans; this chapter starts one level lower, at the scalar condition that decides which reduction or epilogue runs.

**Rule.**

- A Python `if` on a `tl.constexpr` is resolved by the JIT; only the taken branch is emitted.
- A Python `if` on a runtime scalar becomes a device branch, uniform across the program.
- A Python `if` on a tensor is rejected; Triton has no per-lane Python control flow.

**Formula.**

- For `MODE: tl.constexpr`: `if MODE == 0: y = x` / `elif MODE == 1: y = max(x, 0)` compiles one body.
- For a per-lane condition: $m_i=[x_i\gt T]$ and $y_i=m_i x_i+(1-m_i)0$.

**Example.**

| item | value |
|---|---|
| $x$ | $(1,2,3,4,5,6)$ |
| $T$ | $3$ |
| $m=[x\gt T]$ | $(\mathrm F,\mathrm F,\mathrm F,\mathrm T,\mathrm T,\mathrm T)$ |
| $y$ | $(0,0,0,4,5,6)$ |

**Where.**

- Compile-time branches select an algorithm or epilogue (identity, ReLU, GELU), a dtype path, or a block size.
- They cost nothing at run time and remove dead code.

**Triton.**

- Annotate the selector: `MODE: tl.constexpr`, then write `if MODE == 0: ...`.
- Pass it by keyword: `kernel[grid](..., MODE=1, BLOCK=4)`.
- For lanes, write `tl.where(x > T, x, 0.0)`; do not use `if`.

**Failure.**

- `if x > T:` on a tile raises a compile-time error.
- A data-dependent branch would force the warp to serialize both arms when lanes disagree.

**Bridge.** A compile-time condition need not choose an expression; it can also choose a loop, which is §17.2.


### 17.2 `tl.static_range()`

**Connection.** Section 17.1 used a `constexpr` to choose an expression; here the same `constexpr` chooses a loop bound.

**Rule.**

- `tl.static_range(start, end, step)` is a compile-time loop.
- The trip count must be a `constexpr`; the compiler emits one copy of the body per iteration.
- The induction variable is a Python integer, not a tensor.

**Formula.**

- Start at $y_0=1$ and iterate $y_{k+1}=y_k x+1$ four times.
- Expansion: $y_4=((((1)x+1)x+1)x+1)x+1=1+x+x^2+x^3+x^4$.
- The `tl.static_range(0,4)` expansion is four copies of the body.

**Example.**

- At $x=2$: $y_4=1+2+4+8+16=31$.
- The loop-carried value is a single accumulator, so unrolling adds no parallelism.

**Where.**

- A small fixed $K$: a 3-tap stencil, a fixed number of Newton steps, four accumulators for instruction-level parallelism, or a small polynomial.
- The unrolled body has no loop-carried branch.

**Triton.**

- `for k in tl.static_range(0, K):` with `K: tl.constexpr`.
- For a dynamic bound use `range`, not `tl.static_range`.

**Failure.**

- A large $K$ (hundreds or thousands) multiplies code size and register pressure, and can spill.
- A runtime $K$ inside `tl.static_range` is a compile error.

**Bridge.** A static range still needs a fixed count; §17.3 removes that restriction.


### 17.3 `range()`

**Connection.** Section 17.2 fixed the trip count at compile time; here the count becomes a runtime scalar while the body stays one copy.

**Rule.**

- `range(start, end, step)` with a runtime scalar bound emits a device loop: one body, a compare, and a back-edge.
- The loop-carried value is an SSA value updated each trip.

**Formula.**

- Start at $y_0=1$ and iterate $y_{k+1}=c y_k$ for a runtime $n$: $y_n=c^n$.
- For $c=1.1$ and $n=4$: $y_4=1.1^4=1.4641$.
- With $n=0$ the same source returns $y_0=1$.

**Example.**

- Long reductions over $K$ blocks.
- Streaming over a dynamic sequence length.
- Grid-stride loops.
- Any trip count that is not known until launch.

**Where.**

- Use `range` when the bound is a runtime scalar.
- Keep the bound an integer scalar; avoid calling `.item()` on a tensor inside the kernel.

**Triton.**

- `for k in range(0, tl.cdiv(K, BLOCK_K)):`
- `for start in range(pid * BLOCK, n, tl.num_programs(0) * BLOCK):`

**Failure.**

- An unbounded or enormous bound hangs the program.
- A bound that is a tensor with more than one element is not a scalar trip count; reduce it to a scalar first.

**Bridge.** Sections 17.2 and 17.3 give the two loop forms; §17.4 compares their instruction counts and register pressure.


### 17.4 Loops and unrolling

**Connection.** Sections 17.2–17.3 defined the static and runtime loops; this section compares what each costs.

**Rule.**

- Unrolling replicates the body and removes the back-edge.
- It reduces branch overhead and exposes independent instructions, at the cost of code size and possibly registers.
- A runtime loop keeps code size constant and pays one branch per trip.

**Formula.**

- Body cost $c_b$, loop test/back-edge $c_j$, $K$ trips.
- Static code: $C_{\text{loop}}=c_b+c_j$ versus $C_{\text{unroll}}=Kc_b$.
- Executed instructions: $I_{\text{loop}}=K(c_b+c_j)$ versus $I_{\text{unroll}}=Kc_b$.

**Example.**

- With $K=4$, $c_b=8$, $c_j=2$: $C_{\text{loop}}=10$ versus $C_{\text{unroll}}=32$.
- Executed: $I_{\text{loop}}=40$ versus $I_{\text{unroll}}=32$.
- Unrolling saves $8$ branch instructions and costs $22$ extra static instructions.

**Where.**

- Unroll small, hot loops with independent iterations: vector accumulators, stencils, small polynomials.
- Keep long, memory-bound loops rolled to protect the instruction cache and registers.

**Triton.**

- `tl.static_range` is explicit unrolling.
- A plain `range` may still be partially unrolled by the compiler, but the source-level distinction is the trip-count type.

**Failure.**

- Unrolling a dependent recurrence does not create parallelism; it just replicates the same chain.
- Unrolling a large loop can spill registers and slow the kernel down.

**Bridge.** The loop decision is one axis; §17.5 adds the other axis, whether the condition is uniform or per-lane.


### 17.5 Conditional execution

**Connection.** Section 17.1 classified conditions; §17.4 showed when a loop is worth unrolling. Here the classification becomes an execution strategy.

**Rule.**

- A **compile-time** condition selects code.
- A **uniform runtime** condition (a scalar such as `pid`, a kernel argument, or a block index) is a device branch that every lane in the program takes together.
- A **per-lane** condition is not a branch at all; it is a predicate tile applied with `tl.where` or a mask.

**Formula.**

- Uniform early exit: `if pid * B >= N: return`.
- Per-lane select: $y_i=m_i a(x_i)+(1-m_i)b(x_i)$, with $m_i=[x_i\gt T]$.

**Example.**

- $x=(1,\dots,6)$, $T=3$, $a(x)=x$, $b(x)=0$.
- $m=(\mathrm F,\mathrm F,\mathrm F,\mathrm T,\mathrm T,\mathrm T)$.
- $y=(0,0,0,4,5,6)$.

**Where.**

- Uniform branches skip whole tiles: empty tail blocks, unsupported dtype paths.
- Per-lane conditions implement ReLU, thresholding, dropout, attention masks, and piecewise activations without divergence.

**Triton.**

- `if pid * BLOCK >= n_elements: return` is uniform.
- For lanes, use `tl.where(cond, a, b)`.
- For memory, use `mask=cond` in `tl.load`/`tl.store`.

**Failure.**

- A branch on a tile is rejected.
- A runtime scalar branch that differs across programs is fine; a branch inside a lane-varying computation is not expressible, so rewrite it as a select.
- Both arms of `tl.where` are evaluated.

**Bridge.** Once the per-lane condition is a select, the next step is to make that select a mask for both values and memory; that is §17.6.


### 17.6 Mask-based execution

**Connection.** Section 17.5 turned a per-lane condition into a select; here the same predicate becomes a mask that guards memory and chooses values.

**Rule.**

- A mask is a Boolean tile with the same shape as the guarded tile (after broadcasting).
- It guards memory: `tl.load(..., mask=m)` and `tl.store(..., mask=m)`.
- It selects values: `tl.where(m, a, b)`.
- The mask does not skip the arithmetic; it suppresses the memory access and chooses the value.

**Formula.**

- Bounds mask: `offs < n_elements`.
- Value mask: $m_i=[x_i\gt T]$ and $y_i=m_i x_i$.

**Example.**

- Partial tile with `BLOCK=4`:

| program | offsets | bounds mask `offs<6` | `x` loaded (`other=0`) |
|---|---|---|---|
| 0 | 0,1,2,3 | T,T,T,T | 1,2,3,4 |
| 1 | 4,5,6,7 | T,T,F,F | 5,6,0,0 |

- Threshold select:

| $i$ | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| $x_i$ | 1 | 2 | 3 | 4 | 5 | 6 |
| $m_i=[x_i\gt 3]$ | F | F | F | T | T | T |
| $y_i=m_i x_i$ | 0 | 0 | 0 | 4 | 5 | 6 |

**Where.**

- Tail handling, padding, variable sequence lengths, causal attention, dropout, and gated activations all use masks.
- A masked load also needs a safe `other` value for lanes that are not read.

**Triton.**

- `mask = offs < n_elements`
- `x = tl.load(p + offs, mask=mask, other=0.0)`
- `y = tl.where(x > T, x, 0.0)`
- `tl.store(p + offs, y, mask=mask)`
- Both doors, load and store, need the mask.

**Failure.**

- A mask of the wrong shape broadcasts unexpectedly or fails.
- Forgetting the store mask writes garbage past the end.
- A dangerous arm (`1/x`, `log(x)`) is evaluated before `tl.where` selects; guard the input, not the output.

**Bridge.** Masks handle one launch; §17.7 shows how the `constexpr` choices that build those masks become separate compiled bodies.


### 17.7 Compile-time specialization

**Connection.** Section 17.1 chose a branch and §17.6 chose a mask; this section makes the choice itself a compilation boundary.

**Rule.**

- A Triton kernel is compiled per **specialization key**.
- The key includes the `constexpr` arguments, the argument dtypes, and compiler options.
- Runtime arguments (pointers, `n_elements`, scalar scales) are not part of the key.
- Every new key pays the JIT cost once and then reuses the cached binary.

**Formula.**

- Key: $\kappa=(\texttt{MODE},\texttt{ACT},\texttt{BLOCK},\text{dtypes})$.
- The launches $(0,0,4)$, $(1,0,4)$, $(0,1,4)$, $(0,0,8)$ produce four binaries.
- The same four keys with $n=17$ and $n=1000$ still produce four.

**Example.**

- `MODE`, `ACT`, and `BLOCK` are `tl.constexpr`.
- `shift` and `n_elements` are runtime.
- The runtime size changes the grid and the masks, not the binary.

**Where.**

- One source serves fp16/bf16/fp32, several activation epilogues, and several block sizes.
- Autotuning explores `constexpr` keys and caches the best.

**Triton.**

- Annotate only what should specialize: `BLOCK: tl.constexpr`, `MODE: tl.constexpr`.
- Keep `n_elements` a normal argument.
- Pass constexprs by keyword so the call site is readable.

**Failure.**

- Passing `n_elements` as a `constexpr` compiles a new kernel for every tensor size (a JIT storm).
- Passing a tensor where a `constexpr` is expected fails at launch.
- A huge cross-product of constexpr flags multiplies compile time and cache size.

**Bridge.** A specialization key fixes the block; §17.8 keeps the block fixed while letting the tensor size vary.


### 17.8 Dynamic vs static shapes

**Connection.** Section 17.7 fixed the `constexpr` block; here the block stays static while the tensor size becomes a runtime argument.

**Rule.**

- Triton tile shapes must be compile-time: `tl.arange(0, BLOCK)` needs a `constexpr` `BLOCK`.
- Tensor **sizes** may be runtime scalars: pass $N$ as an argument, compute the grid on the host as $\lceil N/B\rceil$, and mask the last tile.
- A static shape is a `constexpr` dimension; a dynamic shape is a runtime $N$ guarded by a mask.

**Formula.**

- Host grid: $\lceil N/B\rceil$.
- Device mask: `offs < n_elements`.
- Kernel signature: `(x_ptr, out_ptr, n_elements, BLOCK: tl.constexpr)`.

**Example.**

- With $B=4$:

| $N$ | grid $\lceil N/B\rceil$ | last offsets | live lanes in last tile |
|---:|---:|---|---:|
| 6 | 2 | 4,5,6,7 | 2 |
| 10 | 3 | 8,9,10,11 | 2 |
| 16 | 4 | 12,13,14,15 | 4 |

- The kernel code is identical for all three; only the grid and the mask change.

**Where.**

- Batch size, sequence length, vocabulary size, and the number of tokens are dynamic in real workloads.
- A static block with a dynamic extent is the standard compromise: one compiled kernel, any $N$.

**Triton.**

- Host grid: `(triton.cdiv(n, BLOCK),)`.
- Device mask: `offs < n_elements`.
- Never write `tl.arange(0, n_elements)`.

**Failure.**

- `tl.arange(0, N)` with runtime `N` is a compile error.
- Assuming $N\bmod B=0$ reads or writes out of bounds.
- Recomputing the grid with a stale $N$ after a shape change leaves the tail unwritten.

**Where this leads.** Parts II–IV turn these eight rules into three methods: scalar reference, kernel controls, and specialization plus dynamic shapes. The worked example next fixes the numbers before the kernels run.


## Chapter map

| § | Topic | Rule | Example | Kernel / demo |
|---|---|---|---|---|
| **17.1** | `if` and compile-time conditions | `constexpr` condition → one emitted branch; tensor condition → mask | `MODE` chooses identity / threshold / select | `mode_mask_kernel` |
| **17.2** | `tl.static_range()` | compile-time trip count → body replicated | $y\leftarrow yx+1$ four times | `poly_static_kernel` |
| **17.3** | `range()` | runtime trip count → one body + back-edge | $y\leftarrow yc$ for $n$ trips | `poly_range_kernel` |
| **17.4** | Loops and unrolling | unrolling trades code size for fewer branches | $C_{\text{loop}}$ vs $C_{\text{unroll}}$ | trade-off model |
| **17.5** | Conditional execution | scalar branch is uniform; lane condition is a select | `if pid*BLOCK >= n: return` | `mode_mask_kernel` |
| **17.6** | Mask-based execution | mask guards memory and selects values | $m=[x\gt 3]$, $y=mx$ | `mode_mask_kernel` |
| **17.7** | Compile-time specialization | new binary per `constexpr` key | `(MODE, ACT, BLOCK)` | `specialize_kernel` |
| **17.8** | Dynamic vs static shapes | static block, runtime $N$, masked tail | $N=6,10,16$ with $B=4$ | `dynamic_scale_kernel` |
| **★** | Grid-stride capstone | one grid handles many tiles | `for start in range(pid*B, n, P*B)` | `grid_stride_kernel` |


## Worked numerical example

### Branch and mask

- Spine: $x=(1,2,3,4,5,6)$, threshold $T=3$.
- Data-dependent branch: keep $x_i$ above $T$, otherwise write zero.
- Mask: $m_i=[x_i\gt T]$.
- Result: $y_i=m_i x_i+(1-m_i)0$.

| $i$ | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| $x_i$ | 1 | 2 | 3 | 4 | 5 | 6 |
| $m_i$ | F | F | F | T | T | T |
| $y_i$ | 0 | 0 | 0 | 4 | 5 | 6 |

- Two-arm select: $a_i=x_i+1=(2,3,4,5,6,7)$ and $b_i=2x_i=(2,4,6,8,10,12)$.
- Selected: $z_i=m_i b_i+(1-m_i)a_i=(2,3,4,8,10,12)$.
- Both arms are evaluated; `tl.where(m, b, a)` only chooses.

### Unrolled loop

- Start $y_0=1$ and iterate $y_{k+1}=y_k x+1$ four times.
- Expansion: $y_1=x+1$, $y_2=x^2+x+1$, $y_3=x^3+x^2+x+1$, $y_4=x^4+x^3+x^2+x+1$.
- `tl.static_range(0,4)` writes four copies of the body.
- At $x=2$: $y_4=16+8+4+2+1=31$.

### Dynamic loop bound

- Start $y_0=1$ and iterate $y_{k+1}=c y_k$ for runtime $n$ trips: $y_n=c^n$.
- $c=1.1$, $n=4$: $y_4=1.1^4=1.4641$.
- $n=0$ returns $1$; $n=8$ returns $1.1^8=2.14358881$; the source does not change.

### Dynamic shape

- With `BLOCK=4`, a dynamic $N$ changes only the grid and the tail mask.
- The kernel code is identical for every row below.

| $N$ | grid | last offsets | tail mask | live |
|---:|---:|---|---|---:|
| 6 | 2 | 4,5,6,7 | T,T,F,F | 2 |
| 10 | 3 | 8,9,10,11 | T,T,F,F | 2 |
| 16 | 4 | 12,13,14,15 | T,T,T,T | 4 |

- The next cell verifies every row with asserts and no printed table.

### Compile-time versus runtime

- `BLOCK`, `MODE`, `ACT`: compile-time; changing one creates a new binary.
- `n_elements`, `threshold`, `shift`, `n_steps`: runtime scalars; changing one does not.
- Pointer arguments: runtime addresses; no new binary.
- Argument dtypes: part of the specialization key.

| quantity | kind | new binary? |
|---|---|---|
| `BLOCK`, `MODE`, `ACT` | `constexpr` | yes |
| `n_elements`, `threshold`, `shift`, `n_steps` | runtime scalar | no |
| pointer arguments | runtime address | no |
| argument dtypes | part of the key | yes |


In [ ]:
# Verify the hand tables without printing them.
x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
T = 3.0
m = x > T
y = torch.where(m, x, torch.zeros_like(x))
a = x + 1.0
b = 2.0 * x
z = torch.where(m, b, a)

assert m.tolist() == [False, False, False, True, True, True]
assert y.tolist() == [0.0, 0.0, 0.0, 4.0, 5.0, 6.0]
assert z.tolist() == [2.0, 3.0, 4.0, 8.0, 10.0, 12.0]

# Unrolled recurrence at x=2: y0=1 and y <- y*2 + 1 four times.
acc = 1.0
for _ in range(4):
    acc = acc * 2.0 + 1.0
assert acc == 31.0
assert 1 + 2 + 4 + 8 + 16 == 31

# Dynamic bound: c=1.1, n=4.
dyn = 1.0
for _ in range(4):
    dyn *= 1.1
assert abs(dyn - 1.1 ** 4) < 1e-12

# Dynamic shape table: (N, grid, live lanes in the last tile).
for n, grid, live in ((6, 2, 2), (10, 3, 2), (16, 4, 4)):
    assert (n + 4 - 1) // 4 == grid
    last = (grid - 1) * 4 + torch.arange(4)
    assert int((last < n).sum()) == live


## Part II — Method 01: scalar branch and Python loop reference

**Connection.** Part I classified conditions; Method 01 implements the same choices one scalar at a time as the correctness oracle.

**Reference shapes.**

- `method01_branch` — one Python `if` per element.
- `method01_select` — compute both arms, then a conditional expression.
- `method01_poly` — a Python loop with fixed or dynamic trip count.
- `method01_loop` — a dynamic trip count with factor `c`.

**Where.**

- On a GPU this is slow: one Python iteration per element.
- On the CPU it is exact and readable, so it anchors the kernel tests.


In [ ]:
def method01_branch(x, threshold):
    """One scalar if per element: keep x_i when it is above the threshold."""
    out = []
    for value in x.tolist():
        if value > threshold:
            out.append(float(value))
        else:
            out.append(0.0)
    return torch.tensor(out, dtype=x.dtype)


def method01_select(x, threshold):
    """Compute both arms, then choose: the scalar analogue of tl.where."""
    out = []
    for value in x.tolist():
        low = float(value) + 1.0
        high = 2.0 * float(value)
        out.append(high if value > threshold else low)
    return torch.tensor(out, dtype=x.dtype)


def method01_poly(value, n_steps):
    """Fixed or dynamic Python loop: y <- y*x + 1, starting at 1."""
    y = 1.0
    for _ in range(n_steps):
        y = y * float(value) + 1.0
    return y


def method01_loop(factor, n_steps):
    """Dynamic Python loop: y <- c*y, starting at 1."""
    y = 1.0
    for _ in range(n_steps):
        y *= float(factor)
    return y


In [ ]:
def test_method01_reference():
    """CPU: the scalar reference matches the hand tables in Part I."""
    x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])
    assert method01_branch(x, 3.0).tolist() == [0.0, 0.0, 0.0, 4.0, 5.0, 6.0]
    assert method01_select(x, 3.0).tolist() == [2.0, 3.0, 4.0, 8.0, 10.0, 12.0]
    assert method01_poly(2.0, 4) == 31.0
    assert method01_poly(2.0, 0) == 1.0
    assert abs(method01_loop(1.1, 4) - 1.1 ** 4) < 1e-12
    assert abs(method01_loop(1.1, 8) - 1.1 ** 8) < 1e-12
    print("method 01: branch, select, static loop, and dynamic loop match the hand tables")


test_method01_reference()


## Part III — Method 02: constexpr mode, static range, and range

**Connection.** Method 01 fixed the values; Method 02 lifts the same branches and loops into Triton kernels.

| Kernel | Control-flow shape | What it demonstrates |
|---|---|---|
| `mode_mask_kernel` | `if MODE == ...` plus `tl.where` | compile-time mode and per-lane mask |
| `poly_static_kernel` | `tl.static_range(0, K)` | unrolled fixed trip count |
| `poly_range_kernel` | `range(0, n_steps)` | runtime trip count |

**Tests.**

- Use the six-element spine and partial tiles (`BLOCK=4`), so the tail mask is exercised in every launch.
- Test functions are collected by the final dispatch cell.
- CPU-only runs print `SKIP`; the interpreter runs them.


In [ ]:
@triton.jit
def mode_mask_kernel(
    x_ptr,
    out_ptr,
    n_elements,
    threshold,
    MODE: tl.constexpr,
    BLOCK: tl.constexpr,
):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    bounds = offs < n_elements
    x = tl.load(x_ptr + offs, mask=bounds, other=0.0)

    # MODE is constexpr: exactly one arm is emitted in the compiled kernel.
    if MODE == 0:
        y = x
    elif MODE == 1:
        y = tl.where(x > threshold, x, 0.0)
    else:
        y = tl.where(x > threshold, 2.0 * x, x + 1.0)

    tl.store(out_ptr + offs, y, mask=bounds)


def run_mode_mask(x, threshold, mode, BLOCK=4):
    out = torch.empty_like(x)
    grid = (triton.cdiv(x.numel(), BLOCK),)
    mode_mask_kernel[grid](x, out, x.numel(), float(threshold),
                           MODE=mode, BLOCK=BLOCK)
    return out


In [ ]:
def test_mode_mask():
    """GPU/interpreter: constexpr MODE and mask-based selection."""
    x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0], device=DEVICE)
    threshold = 3.0

    torch.testing.assert_close(run_mode_mask(x, threshold, 0), x)
    torch.testing.assert_close(
        run_mode_mask(x, threshold, 1),
        torch.where(x > threshold, x, torch.zeros_like(x)),
    )
    torch.testing.assert_close(
        run_mode_mask(x, threshold, 2),
        torch.where(x > threshold, 2.0 * x, x + 1.0),
    )

    spine = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0], device=DEVICE)
    assert run_mode_mask(spine, 3.0, 1, BLOCK=4).tolist() == [0.0, 0.0, 0.0, 4.0, 5.0, 6.0]
    print("mode/mask: three constexpr MODEs and the lane mask agree with the table")


In [ ]:
@triton.jit
def poly_static_kernel(
    x_ptr,
    out_ptr,
    n_elements,
    K: tl.constexpr,
    BLOCK: tl.constexpr,
):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    bounds = offs < n_elements
    x = tl.load(x_ptr + offs, mask=bounds, other=0.0)

    y = tl.full((BLOCK,), 1.0, tl.float32)
    for _ in tl.static_range(0, K):
        y = y * x + 1.0

    tl.store(out_ptr + offs, y, mask=bounds)


def run_poly_static(x, k, BLOCK=4):
    out = torch.empty_like(x)
    grid = (triton.cdiv(x.numel(), BLOCK),)
    poly_static_kernel[grid](x, out, x.numel(), K=k, BLOCK=BLOCK)
    return out


In [ ]:
def test_static_poly():
    """GPU/interpreter: four static iterations equal 1 + x + ... + x^4."""
    x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0], device=DEVICE)
    ref = 1.0 + x + x ** 2 + x ** 3 + x ** 4
    torch.testing.assert_close(run_poly_static(x, 4, BLOCK=4), ref)

    two = torch.tensor([2.0], device=DEVICE)
    assert run_poly_static(two, 4, BLOCK=1).item() == 31.0
    assert run_poly_static(two, 0, BLOCK=1).item() == 1.0
    print("static_range: K=4 expands to 1 + x + x^2 + x^3 + x^4 (x=2 -> 31)")


In [ ]:
@triton.jit
def poly_range_kernel(
    x_ptr,
    out_ptr,
    n_elements,
    n_steps,
    BLOCK: tl.constexpr,
):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    bounds = offs < n_elements
    x = tl.load(x_ptr + offs, mask=bounds, other=0.0)

    y = tl.full((BLOCK,), 1.0, tl.float32)
    for _ in range(0, n_steps):
        y = y * x + 1.0

    tl.store(out_ptr + offs, y, mask=bounds)


def run_poly_range(x, n_steps, BLOCK=4):
    out = torch.empty_like(x)
    grid = (triton.cdiv(x.numel(), BLOCK),)
    poly_range_kernel[grid](x, out, x.numel(), n_steps, BLOCK=BLOCK)
    return out


In [ ]:
def test_range_poly():
    """GPU/interpreter: a runtime trip count matches the Python loop."""
    x = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0], device=DEVICE)

    for n_steps in (0, 1, 4):
        ref = torch.ones_like(x)
        for _ in range(n_steps):
            ref = ref * x + 1.0
        torch.testing.assert_close(run_poly_range(x, n_steps, BLOCK=4), ref)

    # The static and runtime versions agree when the trip counts agree.
    torch.testing.assert_close(
        run_poly_range(x, 4, BLOCK=4),
        run_poly_static(x, 4, BLOCK=4),
    )
    print("range: runtime trip counts 0, 1, and 4 match the Python loop")


## Part IV — Method 03: specialization and dynamic shapes

**Connection.** Method 02 used one control-flow shape per kernel; Method 03 composes `constexpr` choices, a runtime size, and a grid-stride loop.

**Specialization kernel.**

- `specialize_kernel` takes `MODE`, `ACT`, and `BLOCK` as `constexpr`.
- `shift` and `n_elements` are runtime.
- Each `(MODE, ACT, BLOCK, dtype)` key compiles a separate body; runtime shift and size do not.

**Dynamic-shape kernel.**

- `dynamic_scale_kernel` takes $N$ at run time.
- Host grid: $\lceil N/B\rceil$.
- The kernel masks the tail, so one binary serves every $N$.

**Grid-stride capstone.**

- `grid_stride_kernel` launches a fixed number of programs.
- Each program loops over tiles: `start = pid*B, pid*B + P*B, ... < N`, where $P$ is the number of programs.
- This avoids launching one block per tile.


In [ ]:
@triton.jit
def specialize_kernel(
    x_ptr,
    out_ptr,
    n_elements,
    shift,
    MODE: tl.constexpr,
    ACT: tl.constexpr,
    BLOCK: tl.constexpr,
):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    bounds = offs < n_elements
    x = tl.load(x_ptr + offs, mask=bounds, other=0.0)

    if MODE == 0:
        z = x + shift
    elif MODE == 1:
        z = x * shift
    else:
        z = x - shift

    if ACT == 0:
        y = z
    elif ACT == 1:
        y = tl.maximum(z, 0.0)
    else:
        y = tl.minimum(z, 1.0)

    tl.store(out_ptr + offs, y, mask=bounds)


def run_specialize(x, shift, mode, act, BLOCK=4):
    out = torch.empty_like(x)
    grid = (triton.cdiv(x.numel(), BLOCK),)
    specialize_kernel[grid](x, out, x.numel(), float(shift),
                            MODE=mode, ACT=act, BLOCK=BLOCK)
    return out


def reference_specialize(x, shift, mode, act):
    if mode == 0:
        z = x + shift
    elif mode == 1:
        z = x * shift
    else:
        z = x - shift
    if act == 0:
        return z
    if act == 1:
        return torch.clamp(z, min=0.0)
    return torch.clamp(z, max=1.0)


In [ ]:
def test_specialization():
    """GPU/interpreter: every constexpr (MODE, ACT) key computes its own body."""
    x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0, 4.0, 5.0], device=DEVICE)
    combos = ((0, 0), (0, 1), (1, 0), (1, 1), (1, 2), (2, 2))

    for mode, act in combos:
        got = run_specialize(x, 2.0, mode, act, BLOCK=4)
        ref = reference_specialize(x, 2.0, mode, act)
        torch.testing.assert_close(got, ref)

    # A (MODE, ACT) pair is a specialization key; the runtime shift and N are not.
    assert len(set(combos)) == len(combos) == 6
    print(f"specialization: {len(combos)} (MODE, ACT) keys compile separate bodies")


In [ ]:
@triton.jit
def dynamic_scale_kernel(
    x_ptr,
    out_ptr,
    n_elements,
    scale,
    BLOCK: tl.constexpr,
):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    bounds = offs < n_elements
    x = tl.load(x_ptr + offs, mask=bounds, other=0.0)
    tl.store(out_ptr + offs, x * scale + 1.0, mask=bounds)


def run_dynamic_scale(x, scale, BLOCK=8):
    out = torch.empty_like(x)
    n = x.numel()
    if n == 0:
        return out
    grid = (triton.cdiv(n, BLOCK),)
    dynamic_scale_kernel[grid](x, out, n, float(scale), BLOCK=BLOCK)
    return out


In [ ]:
def test_dynamic_shape():
    """GPU/interpreter: one binary handles every runtime N and masks the tail."""
    for n in (1, 7, 16, 1000):
        x = torch.arange(n, dtype=torch.float32, device=DEVICE)
        got = run_dynamic_scale(x, 3.0, BLOCK=8)
        torch.testing.assert_close(got, 3.0 * x + 1.0)

    # N=0 is a host-side no-op, not a kernel launch.
    empty = torch.empty(0, device=DEVICE)
    assert run_dynamic_scale(empty, 3.0, BLOCK=8).numel() == 0

    # The grid is a host-side function of N; the tail mask is a device-side function of N.
    for n, grid, tail in ((1, 1, 1), (7, 1, 7), (16, 2, 8), (1000, 125, 8)):
        assert triton.cdiv(n, 8) == grid
        assert min(8, n - (grid - 1) * 8) == tail
    print("dynamic shape: N=1, 7, 16, 1000 and N=0 all match; only grid and mask change")


In [ ]:
@triton.jit
def grid_stride_kernel(
    x_ptr,
    out_ptr,
    n_elements,
    BLOCK: tl.constexpr,
):
    pid = tl.program_id(0)
    n_programs = tl.num_programs(0)
    stride = n_programs * BLOCK

    for start in range(pid * BLOCK, n_elements, stride):
        offs = start + tl.arange(0, BLOCK)
        bounds = offs < n_elements
        x = tl.load(x_ptr + offs, mask=bounds, other=0.0)
        tl.store(out_ptr + offs, 2.0 * x + 1.0, mask=bounds)


def run_grid_stride(x, grid_size, BLOCK=8):
    out = torch.empty_like(x)
    grid_stride_kernel[(grid_size,)](x, out, x.numel(), BLOCK=BLOCK)
    return out


In [ ]:
def test_grid_stride():
    """GPU/interpreter: a fixed grid loops over many tiles without losing the tail."""
    x = torch.arange(37, dtype=torch.float32, device=DEVICE)

    # grid=2 < cdiv(37, 8)=5: each program takes several strides.
    assert 2 < triton.cdiv(37, 8)
    torch.testing.assert_close(run_grid_stride(x, 2, BLOCK=8), 2.0 * x + 1.0)

    # A full grid is also correct; each program takes exactly one trip.
    torch.testing.assert_close(run_grid_stride(x, 5, BLOCK=8), 2.0 * x + 1.0)

    # Every element is written exactly once by the two-program launch.
    got = run_grid_stride(x, 2, BLOCK=8)
    assert got.numel() == 37 and torch.isfinite(got).all()
    print("grid-stride: 2 programs cover 37 elements; 5 programs give the same result")


## Unrolling versus loops

**Connection.** Sections 17.2–17.3 defined the static and runtime loop; this section compares their cost.

**Trade.**

- Dynamic instruction count: `I_rolled = K(c_b+c_j)` vs `I_unrolled = K c_b`.
- Static code size: `C_rolled = c_b+c_j` vs `C_unrolled = K c_b`.
- Branch saving: `K c_j`.
- Code-size multiplier: `K c_b/(c_b+c_j)`.

**Example ($c_b=8$, $c_j=2$).**

| $K$ | static rolled | static unrolled | executed rolled | executed unrolled | branch saving | code multiplier |
|---:|---:|---:|---:|---:|---:|---:|
| 4 | 10 | 32 | 40 | 32 | 8 | 3.2× |
| 8 | 10 | 64 | 80 | 64 | 16 | 6.4× |
| 32 | 10 | 256 | 320 | 256 | 64 | 25.6× |
| 256 | 10 | 2048 | 2560 | 2048 | 512 | 204.8× |

**Register pressure.**

- Independent iterations can keep several accumulators live and hide latency.
- Dependent iterations (the recurrence $y\leftarrow yx+1$) form one chain, so unrolling adds instructions but not parallelism.
- A large unroll can exceed the register file and spill to local memory.

**Code size.**

- The instruction cache is finite; a 2048-instruction body may evict the rest of the kernel.
- A rolled loop keeps the hot path small.

**Rule of thumb.**

- Unroll a small, hot, independent loop (typically $K\le 8$).
- Leave long, memory-bound, or dependent loops rolled.
- Measure; do not guess from the table alone.


In [ ]:
def rolled_cost(k, body=8, branch=2):
    return k * (body + branch), body + branch


def unrolled_cost(k, body=8, branch=2):
    return k * body, k * body


ks = (4, 8, 32, 256)
executed_rolled = [rolled_cost(k)[0] for k in ks]
executed_unrolled = [unrolled_cost(k)[0] for k in ks]
static_rolled = [rolled_cost(k)[1] for k in ks]
static_unrolled = [unrolled_cost(k)[1] for k in ks]

assert executed_rolled == [40, 80, 320, 2560]
assert executed_unrolled == [32, 64, 256, 2048]
assert static_rolled == [10, 10, 10, 10]
assert static_unrolled == [32, 64, 256, 2048]
assert [r - u for r, u in zip(executed_rolled, executed_unrolled)] == [8, 16, 64, 512]

print("unroll model: executed instructions drop by K*c_j; static code grows by K*c_b")


## Mask-based execution versus a divergent branch

**Connection.** Section 17.6 established masks; this section compares a mask with a real branch when lanes disagree.

**Cost.**

- Divergent branch: `C_divergent = c_a + c_b`.
- Mask: `C_mask = c_a + c_b + c_select`.
- Both arms are evaluated by `tl.where`; a uniform branch can skip one.

**When to use.**

| Situation | Best choice | Why |
|---|---|---|
| condition uniform per program (`pid`, `n_elements`) | scalar `if` / early `return` | whole program skips the arm |
| condition varies per lane, both arms cheap | `tl.where` | one converged stream, no divergence |
| condition varies per lane, one arm expensive | restructure or split the kernel | both arms would be computed by `tl.where` |
| condition guards memory | `mask=` on load/store | no out-of-bounds access, no branch |
| condition selects a whole algorithm | `tl.constexpr` | the dead algorithm is not compiled |

**Key point.**

- `tl.where` is not lazy; it evaluates both arms for every lane and then selects.
- If an arm can divide by zero, overflow, or read an invalid address, guard the input of that arm, not the output.


In [ ]:
# A mask computes both arms; a scalar branch computes one. Model the cost.
def mask_cost(a=3, b=5, select=1):
    return a + b + select


def branch_cost(a=3, b=5):
    return a + b


assert mask_cost() == 9
assert branch_cost() == 8

x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0])
m = x > 0
selected = torch.where(m, 2.0 * x, x + 1.0)
assert selected.tolist() == [-1.0, 0.0, 1.0, 2.0, 4.0]

# Guarding the input is different from selecting the output.
safe = torch.where(m, 2.0 * x, torch.zeros_like(x))
assert safe.tolist() == [0.0, 0.0, 0.0, 2.0, 4.0]
print("mask vs branch: both arms are evaluated; the select adds one instruction")


## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| `if x > T:` on a tile | compile error; there is no per-lane Python branch | `tl.where(x > T, ...)` or a mask |
| `and` / `or` / `not` on tensors | raises or short-circuits a scalar | `&`, `\|`, `~` with parentheses |
| `constexpr` on a runtime value | new binary per value; JIT storm | keep `n_elements` a normal argument |
| `tl.static_range` with a runtime bound | compile error | `range(0, n_steps)` |
| unbounded `range` / dynamic bound | hang or out-of-bounds loop | clamp the bound; test `n=0`, `n=1`, `n=N` |
| dynamic shape as `constexpr` | recompiles for every $N$ | static `BLOCK`, runtime `n_elements`, tail mask |
| mask shape mismatch | broadcast error or silently wrong lanes | mask shape = guarded tile shape; test the tail |
| unrolling a large loop | code bloat, register spills, i-cache misses | unroll small independent loops only |
| forgetting the store mask | writes past the allocation | mask both `tl.load` and `tl.store` |
| `tl.where` with a dangerous arm | `inf`/`nan` computed before the select | guard the input of the arm |


## Self-check

1. For $x=(1,2,3,4,5,6)$ and $T=3$, write $[x\gt T]$ and the result of `tl.where(x>T, x, 0)`.
2. Explain the difference between `if MODE == 0` with `MODE: tl.constexpr` and `if pid == 0`.
3. Why can a tensor condition not be a Python `if`? What replaces it?
4. Write the expansion of `tl.static_range(0,4)` for $y\leftarrow yx+1$ starting at $y_0=1$.
5. For $c_b=8$, $c_j=2$, $K=8$, compute static and executed instruction counts for the rolled and unrolled versions.
6. Which arguments create a new specialization? Which do not?
7. With `BLOCK=4`, $N=10$, give the grid, the last offsets, and the tail mask.
8. Why is `tl.where` not a short-circuit? Give an example where that matters.
9. Rewrite `if x_i > 0: y_i = 1/x_i else: y_i = 0` safely.


## Summary

**Thread.** Control flow in Triton is a type decision, not a style choice.

- A condition on a `tl.constexpr` is resolved at compile time; only one branch is emitted.
- A condition on a runtime scalar is a uniform device branch (or an early `return`).
- A condition on a tile is a predicate tile; use `tl.where` for values and `mask=` for memory.
- `tl.static_range` unrolls a fixed trip count; `range` keeps a runtime loop.
- Unrolling trades code size and register pressure for fewer branches and more ILP.
- `tl.where` evaluates both arms; it does not skip the expensive one.
- Each distinct `constexpr` key compiles a new binary; runtime sizes do not.
- A static `BLOCK` plus a runtime $N$ plus a tail mask is the standard dynamic-shape pattern.
- A grid-stride loop lets a fixed grid cover a dynamic number of tiles.

**The one-sentence rule.** If the compiler can know it, branch; if only the lane can know it, mask; if only the launch can know it, pass it as a runtime scalar and mask the tail.

**Where this leads.** The exercises extend each rule: scalar-to-mask, unroll factor, dynamic bound, specialization count, grid-stride, safe select, and masked reduction.


## Exercises

1. **Scalar to mask.** Rewrite `method01_branch` as a Triton kernel `y = tl.where(x > T, x, 0)`
   with `T` a runtime scalar. Test `T=0`, `T=3`, and `T=10` on the spine.
2. **Three-mode epilogue.** Extend `mode_mask_kernel` with `MODE=3` for
   $y=\operatorname{sign}(x)\sqrt{\lvert x\rvert}$ (a signed square root). Test both signs.
3. **Unroll factor.** Run `poly_static_kernel` with `K=1,2,4,8,16` and compare the number of
   compiled instructions (inspect the generated PTX/SASS if available). Where does it spill?
4. **Dynamic bound.** Write a kernel that accumulates `y <- y*1.1` exactly `n_steps` times,
   with `n_steps` a runtime argument. Check `n_steps=0,1,4,8` against Python.
5. **Partial tile.** For `BLOCK=4`, write the grid, last offsets, and tail mask for
   $N=6,7,8,9,10$. Which two $N$ values have the same grid but different masks?
6. **Specialization count.** Launch `specialize_kernel` with the six `(MODE, ACT)` pairs and
   then with `n=17` and `n=1000`. How many binaries are compiled? Justify the count.
7. **Grid-stride.** Change `grid_stride_kernel` to use `n_programs=1` and `n=37`. How many
   loop trips does the single program take? Verify the output.
8. **Uniform early exit.** Add `if pid * BLOCK >= n_elements: return` to a kernel. Prove that
   it is uniform and that removing it does not change the output (only the wasted work).
9. **Safe select.** Implement
   $y=\texttt{tl.where}(x\ne0,\,1/x,\,0)$ so that it never creates `inf` or `nan`. Test
   $x=(0,1,-1,2,-2)$.
10. **Masked reduction.** Use a bounds mask and `other=-inf` to compute the maximum of a
    partial last tile. Why is `other=0` wrong when all live values are negative?


In [ ]:
print("CPU reference checks ...")
test_method01_reference()

print("\nGPU/interpreter checks ...")
for fn in (
    test_mode_mask,
    test_static_poly,
    test_range_poly,
    test_specialization,
    test_dynamic_shape,
    test_grid_stride,
):
    run_if_gpu(fn)


## Notes and caveats

- The tests run under the Triton interpreter (`TRITON_INTERPRET=1`), which executes kernels
  on the CPU. It checks shapes, masks, and values, not performance.
- `tl.num_programs(0)` and runtime `range` bounds are supported by the interpreter in the
  pinned environment; on a real GPU they lower to the expected grid-stride loop.
- Compilation counts are described by the cache-key model. The exact number of binaries also
  depends on argument dtypes and compiler options; inspect the JIT cache if you need the
  precise count.
- The instruction-count model uses $c_b=8$ and $c_j=2$ as a teaching example. Real counts come
  from the generated PTX/SASS and vary by architecture.
- `tl.where` is eager: both arms are evaluated. A branch on a uniform scalar can still skip
  work that a select cannot.
